### Экспепримент 2. Кластеризация векторов

In [ ]:
import os
import pandas as pd

EMBEDDINGS_DIR = "/content"

In [ ]:
def safe_filename(text: str) -> str:
    return (
        str(text)
        .replace("/", "_")
        .replace("\\", "_")
        .strip()
    )

In [ ]:
def make_qwen_embedding_path(verb: str) -> str:
    return os.path.join(EMBEDDINGS_DIR, f"{safe_filename(verb)}.npy")

def make_mistral_embedding_path(verb: str) -> str:
    return os.path.join(EMBEDDINGS_DIR, f"{safe_filename(verb)}_mistral.npy")

In [ ]:
qwen_df = pd.read_csv('qwen.csv')
mistral_df = pd.read_csv('mistral.csv')

In [ ]:
qwen_df["embedding_path"] = qwen_df["verb"].apply(make_qwen_embedding_path)
mistral_df["embedding_path"] = mistral_df["verb"].apply(make_mistral_embedding_path)

In [ ]:
import numpy as np

def load_embedding_matrix(df):
    vectors = []
    for path in df["embedding_path"]:
        vec = np.load(path)
        vectors.append(vec)
    return np.vstack(vectors)

X_qwen = load_embedding_matrix(qwen_df)
X_mistral = load_embedding_matrix(mistral_df)

(X_qwen.shape)
print(X_mistral.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

sim_qwen = cosine_similarity(X_qwen)
sim_mistral = cosine_similarity(X_mistral)

sim_qwen_df = pd.DataFrame(sim_qwen, index=qwen_df["verb"], columns=qwen_df["verb"])
sim_mistral_df = pd.DataFrame(sim_mistral, index=mistral_df["verb"], columns=mistral_df["verb"])

sim_qwen_df.iloc[:5, :5]
sim_mistral_df.iloc[:5, :5]

In [ ]:
import matplotlib.pyplot as plt

def plot_similarity_heatmap(sim_df, title):
    plt.figure(figsize=(10, 8))
    plt.imshow(sim_df.values, aspect="auto", cmap="gray")
    plt.colorbar(label="Cosine similarity")
    plt.xticks(range(len(sim_df.columns)), sim_df.columns, rotation=90)
    plt.yticks(range(len(sim_df.index)), sim_df.index)
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_similarity_heatmap(sim_qwen_df, "Qwen similarity matrix")
plot_similarity_heatmap(sim_mistral_df, "Mistral similarity matrix")

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

def run_hierarchical_clustering(df, X, model_name, n_clusters=4):
    dist = pdist(X, metric="cosine")
    Z = linkage(dist, method="average")

    plt.figure(figsize=(14, 6))
    dendrogram(Z, labels=df["verb"].tolist(), leaf_rotation=90)
    plt.title(f"Hierarchical clustering: {model_name}")
    plt.tight_layout()
    plt.show()

    cluster_ids = fcluster(Z, t=n_clusters, criterion="maxclust")

    out = df.copy()
    out["cluster_id"] = cluster_ids
    return out, Z

qwen_clustered, Z_qwen = run_hierarchical_clustering(qwen_df, X_qwen, "qwen", n_clusters=4)
mistral_clustered, Z_mistral = run_hierarchical_clustering(mistral_df, X_mistral, "mistral", n_clusters=4)

In [ ]:
from sklearn.decomposition import PCA

def plot_pca(df, X, title):
    pca = PCA(n_components=2)
    coords = pca.fit_transform(X)

    plot_df = df.copy()
    plot_df["x"] = coords[:, 0]
    plot_df["y"] = coords[:, 1]

    plt.figure(figsize=(10, 8))
    for cluster_id in sorted(plot_df["cluster_id"].unique()):
        sub = plot_df[plot_df["cluster_id"] == cluster_id]
        plt.scatter(sub["x"], sub["y"], label=f"Cluster {cluster_id}")
        for _, row in sub.iterrows():
            plt.text(row["x"], row["y"], row["verb"], fontsize=9)

    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_pca(qwen_clustered, X_qwen, "Qwen PCA")
plot_pca(mistral_clustered, X_mistral, "Mistral PCA")

In [ ]:
def cluster_suffix_analysis(clustered_df):
    rows = []

    for cluster_id, sub in clustered_df.groupby("cluster_id"):
        suffix_counts = sub["suffix"].value_counts(dropna=False)
        dominant_suffix = suffix_counts.index[0]
        dominant_count = suffix_counts.iloc[0]
        dominant_share = dominant_count / len(sub)

        rows.append({
            "cluster_id": cluster_id,
            "n_items": len(sub),
            "n_distinct_suffixes": sub["suffix"].nunique(dropna=True),
            "dominant_suffix": dominant_suffix,
            "dominant_suffix_count": int(dominant_count),
            "dominant_suffix_share": round(dominant_share, 3),
            "verbs": "; ".join(sub["verb"].astype(str).tolist()),
            "suffixes": "; ".join(sub["suffix"].astype(str).tolist()),
        })

    return pd.DataFrame(rows).sort_values("cluster_id")

qwen_cluster_suffix = cluster_suffix_analysis(qwen_clustered)
mistral_cluster_suffix = cluster_suffix_analysis(mistral_clustered)
display(qwen_cluster_suffix)
display(mistral_cluster_suffix)

In [ ]:
def suffix_coherence_summary(cluster_suffix_df, model_name):
    return pd.DataFrame([{
        "model": model_name,
        "mean_dominant_suffix_share": cluster_suffix_df["dominant_suffix_share"].mean(),
        "mean_n_distinct_suffixes": cluster_suffix_df["n_distinct_suffixes"].mean(),
        "num_clusters": len(cluster_suffix_df),
    }])

embedding_summary = pd.concat([
    suffix_coherence_summary(qwen_cluster_suffix, "qwen"),
    suffix_coherence_summary(mistral_cluster_suffix, "mistral")
], ignore_index=True)

embedding_summary